In [1]:
import os

os.environ["GOOGLE_API_KEY"] = "your_gemini_api_key_here"
os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"

In [2]:
# Check and display a masked version of the GEMINI_API_KEY without revealing the secret
key = os.environ.get("GEMINI_API_KEY")
if not key:
    print("GEMINI_API_KEY is not set")
else:
    if len(key) <= 8:
        masked = "*" * len(key)
    else:
        masked = key[:4] + "*" * (len(key) - 8) + key[-4:]
    print("GEMINI_API_KEY is set. Masked value:", masked)

GEMINI_API_KEY is set. Masked value: your****************here


Stream and Batch


In [3]:
import requests

def get_latest_weather(city: str):
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_resp = requests.get(
        geo_url,
        params={"name": city, "count": 1, "language": "en", "format": "json"},
        timeout=10,
    )
    geo_data = geo_resp.json()

    if "results" not in geo_data or not geo_data["results"]:
        return f"City not found: {city}"

    place = geo_data["results"][0]
    lat, lon = place["latitude"], place["longitude"]

    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_resp = requests.get(
        weather_url,
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,wind_speed_10m,weather_code",
            "timezone": "auto",
        },
        timeout=10,
    )
    weather_data = weather_resp.json()

    current = weather_data.get("current", {})
    return {
        "city": place["name"],
        "country": place.get("country"),
        "time": current.get("time"),
        "temperature_c": current.get("temperature_2m"),
        "wind_speed_kmh": current.get("wind_speed_10m"),
        "weather_code": current.get("weather_code"),
    }

print(get_latest_weather("London"))

{'city': 'London', 'country': 'United Kingdom', 'time': '2026-05-10T09:30', 'temperature_c': 12.3, 'wind_speed_kmh': 18.4, 'weather_code': 3}


### Define and Invoke the Model

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

# Create your .env path and load it. Use override=True to replace the invalid keys 
# that were set to 'your_gemini_api_key_here' in an earlier cell.
env_path = r"C:\Langchain\.env"
load_dotenv(dotenv_path=env_path, override=True)

# Initialize the Gemini model
model = ChatGoogleGenerativeAI(model="gemini-flash-latest")

# Now you can use model.invoke()
response = model.invoke("Hello, what can you do?")
print(response.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-flash-latest' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}

### Steam output

In [9]:
model.stream("Write me a 500 word essay on the history of the internet")

<generator object BaseChatModel.stream at 0x000001EB8A28A700>

In [10]:
for chunk in model.stream("Write me a 500 word essay on the history of the internet"):
    if chunk.text:
        print(chunk.text, end="", flush=True)   

The internet is arguably the most significant technological achievement of the 20th century. What began as a niche military research project has evolved into a global infrastructure that underpins modern civilization, transforming how we communicate, work, and perceive the world. The history of the internet is not a single "eureka" moment, but rather a decades-long evolution driven by collaboration, standardization, and visionary thinking.

The foundations of the internet were laid during the Cold War in the late 1950s and 1960s. Fearing a nuclear strike that could dismantle centralized communication systems, the United States Department of Defense created the Advanced Research Projects Agency (ARPA). The goal was to develop a decentralized network that could survive a partial outage. This led to the creation of ARPANET in 1969. The first message ever sent across this network—from UCLA to Stanford—was supposed to be the word "LOGIN," but the system crashed after the first two letters, 

In [11]:
model.invoke("why do parrots have colorful feathers?")

AIMessage(content=[{'type': 'text', 'text': 'Parrots are among the most vibrant creatures on Earth, and their brilliant plumage isn\'t just for show. Their colors are the result of millions of years of evolution, serving several critical biological and survival functions.\n\nHere are the primary reasons why parrots have colorful feathers:\n\n### 1. Camouflage (Disruptive Coloration)\nIt might seem counterintuitive that a bright red or yellow bird could be "camouflaged," but in their natural habitats—tropical rainforests—these colors act as excellent cover.\n*   **Dappled Light:** Rainforests are filled with bright sunlight filtering through green leaves, colorful flowers, and ripe fruit. A bright green parrot blends perfectly with the canopy, while splashes of red or yellow mimic tropical blooms or spots of sunlight.\n*   **Breaking up the Outline:** Bold patterns and bright colors help break up the bird’s physical outline, making it harder for predators (like hawks or monkeys) to reco

In [12]:
for chunk in model.stream("Write me a 100 word essay on the history of the BJP"):
    if chunk.text:
        print(chunk.text, end=" |", flush=True)

The Bharatiya Janata Party (BJP) traces its roots to the Bharatiya Jana Sangh, founded in 1951 | by Syama Prasad Mukherjee. After merging into the Janata Party post-Emergency, it reconstituted as the BJP in 1 |980 under Atal Bihari Vajpayee and L.K. Advani. Initially focusing on "Gandh |ian Socialism," the party soon pivoted toward Hindutva, gaining national prominence during the 1990s Ram | Janmabhoomi movement. The BJP led the country first in 1996, then from 1 |998 to 2004. Since 2014, under Narendra Modi’s leadership, | the party has achieved unprecedented electoral dominance, transforming India’s political landscape with a focus on nationalism and development. |

In [13]:
for chunk in model.stream("Hii how are you "):
        print(chunk.text, end=" * ", flush=True)

Hello! I'm doing well, thank you for asking * . How are you today? Is there anything I can help you with? *  *  * 

#### BATCh

In [15]:
responses = model.batch([
    "why do parrots have colorful feathers?",
    "Write me a 100 word essay on the history of the BJP",  
    "Why congress are losing elections in India?"
])

for resp in responses:
    print(resp.content)
    print("-" * 50)

[{'type': 'text', 'text': 'The vibrant colors of parrots are not just for show; they are the result of millions of years of evolution. Their feathers serve several critical biological purposes, ranging from survival to social interaction.\n\nHere are the primary reasons why parrots have colorful feathers:\n\n### 1. Camouflage (The "Canopy Effect")\nIt might seem counterintuitive that a bright red or yellow bird could be "hidden," but in their natural habitat—the tropical rainforest—bright colors provide excellent camouflage.\n*   **Green feathers:** Most parrots are primarily green, which allows them to blend in perfectly with the lush foliage of the canopy.\n*   **Bright spots:** The splashes of red, yellow, and blue mimic the dappled sunlight, colorful tropical flowers, and ripening fruit found in the trees. This "disruptive coloration" breaks up the bird\'s outline, making it harder for predators like hawks or eagles to spot them from above.\n\n### 2. Mating and Sexual Selection\nIn

In [21]:
responses = model.batch([
    "why do parrots have colorful feathers?",
    "Write me a 100 word essay on the history of the BJP",  
    "Why congress are losing elections in India?"],
    config={'max_Concurrency': 5},
)
for resp in responses:
    print(resp.content)
    print("-" * 50)

[{'type': 'text', 'text': 'Parrots are among the most vibrant creatures on Earth, and their brilliant colors are not just for show. Their feathers serve several critical evolutionary purposes, ranging from survival to social communication.\n\nHere are the primary reasons why parrots have colorful feathers:\n\n### 1. Camouflage (Disruptive Coloration)\nIt might seem counterintuitive that a bright red or yellow bird could be "hidden," but in their natural habitats—tropical rainforests—bright colors provide excellent camouflage.\n*   **Green feathers:** Most parrots are primarily green, which allows them to blend perfectly into the lush canopy of leaves.\n*   **Bright spots:** The splashes of red, yellow, and blue mimic the "dappled" sunlight hitting tropical flowers, fruits, and shadows. This is called **disruptive coloration**, which breaks up the bird\'s outline and makes it harder for predators (like hawks or monkeys) to spot them among the foliage.\n\n### 2. Mating and Sexual Selecti

In [1]:
model.invoke("how many r in Strawberry")

NameError: name 'model' is not defined